In [1]:
# validate the downscaled data using the same framework with all raw datasets

# V1 - 2026-09-25 - create code - jpk
# V2 - 2026-10-12 - updates for final run 

In [ ]:
# libraries
import pandas as pd
import numpy as np
import glob
import os
import rasterio
import HydroErr as he
import concurrent.futures
from pyproj import Transformer

#paths
insitu_tmax_raw = pd.read_parquet(r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Insitu\Cleaned\tmax_cleaned.parquet")
insitu_tmin_raw = pd.read_parquet(r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Insitu\Cleaned\tmin_cleaned.parquet")

tmax_train_df = r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmax_train_stations.csv"
tmax_test_df = r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmax_test_stations.csv"
tmin_train_df = r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmin_train_stations.csv"
tmin_test_df = r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmin_test_stations.csv"

output_dir = fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\Validation_results"
os.makedirs(output_dir, exist_ok=True)

#parameters
datasets = ['ERA5L', 'HAR', 'CHIRTS', 'CHELSA', 'IK', 'DS']
start_year, start_month = 2001, 1
end_year,   end_month   = 2025, 9
start_date = pd.Timestamp(f"{start_year}-{start_month:02d}-01")
end_date   = pd.Timestamp(f"{end_year}-{end_month:02d}-01") + pd.offsets.MonthEnd(0)
min_valid_days = 20

In [3]:
# 2 Aggregate insitu monthly with completeness filter

print("==========Aggregating insitu tmax to monthly totals==============")
insitu_tmax_raw["date"] = pd.to_datetime(insitu_tmax_raw[['year', 'month', 'day']])
insitu_tmax_raw = insitu_tmax_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'max_temp': 'tmax'})
insitu_tmax_raw = insitu_tmax_raw[insitu_tmax_raw["tmax"] > -9999]

insitu_tmax_monthly = (
    insitu_tmax_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        tmax_average = ('tmax', 'mean'),
        valid_days = ('tmax', 'count')
    )
    .reset_index()
)

n_before_tmax = len(insitu_tmax_monthly)
insitu_tmax_monthly = insitu_tmax_monthly[insitu_tmax_monthly["valid_days"] >= min_valid_days].copy()
n_after_tmax = len(insitu_tmax_monthly)
print(f"    Station-months before completeness filter: {n_before_tmax:,}")
print(f"    Station-months after (>= {min_valid_days} valid days): {n_after_tmax:,}")
print(f"    Excluded: {n_before_tmax - n_after_tmax:,} ({100*(n_before_tmax-n_after_tmax)/n_before_tmax:.1f}%)")

# Filter to analysis period
insitu_tmax_monthly = insitu_tmax_monthly[
    (insitu_tmax_monthly["year"]    >=  start_year) &
    (insitu_tmax_monthly["year"]    <=  end_year)
].copy()

insitu_tmax_monthly[["x", "y"]] = insitu_tmax_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_monthly_train = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_monthly_test = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(test_ids_tmax)].copy()

unique_stn_tmax = (
    insitu_tmax_monthly_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmax = list(zip(unique_stn_tmax['x'], unique_stn_tmax['y']))  

print(f"\n  Train station-months: {len(insitu_tmax_monthly_train):,}")
print(f"  Test  station-months: {len(insitu_tmax_monthly_test):,}")

print("\n==========Aggregating insitu tmin to monthly totals==============")
insitu_tmin_raw["date"] = pd.to_datetime(insitu_tmin_raw[['year', 'month', 'day']])
insitu_tmin_raw = insitu_tmin_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'min_temp': 'tmin'})
insitu_tmin_raw = insitu_tmin_raw[insitu_tmin_raw["tmin"] > -9999]

insitu_tmin_monthly = (
    insitu_tmin_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        tmin_average = ('tmin', 'mean'),
        valid_days = ('tmin', 'count')
    )
    .reset_index()
)

n_before_tmin = len(insitu_tmin_monthly)
insitu_tmin_monthly = insitu_tmin_monthly[insitu_tmin_monthly["valid_days"] >= min_valid_days].copy()
n_after_tmin = len(insitu_tmin_monthly)
print(f"    Station-months before completeness filter: {n_before_tmin:,}")
print(f"    Station-months after (>= {min_valid_days} valid days): {n_after_tmin:,}")
print(f"    Excluded: {n_before_tmin - n_after_tmin:,} ({100*(n_before_tmin-n_after_tmin)/n_before_tmin:.1f}%)")

# Filter to analysis period
insitu_tmin_monthly = insitu_tmin_monthly[
    (insitu_tmin_monthly["year"]    >=  start_year) &
    (insitu_tmin_monthly["year"]    <=  end_year)
].copy()

insitu_tmin_monthly[["x", "y"]] = insitu_tmin_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_monthly_train = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_monthly_test = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(test_ids_tmin)].copy()

unique_stn_tmin = (
    insitu_tmin_monthly_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmin = list(zip(unique_stn_tmin['x'], unique_stn_tmin['y']))  

print(f"\n  Train station-months: {len(insitu_tmin_monthly_train):,}")
print(f"  Test  station-months: {len(insitu_tmin_monthly_test):,}")


==========Aggregating insitu tmax to monthly totals==============
    Station-months before completeness filter: 14,894
    Station-months after (>= 20 valid days): 14,804
    Excluded: 90 (0.6%)

  Train station-months: 11,310
  Test  station-months: 3,494

==========Aggregating insitu tmin to monthly totals==============
    Station-months before completeness filter: 15,073
    Station-months after (>= 20 valid days): 14,983
    Excluded: 90 (0.6%)

  Train station-months: 11,560
  Test  station-months: 3,423


In [4]:
#Helper

def extract_date(filepath, dataset):
    """Parse date from filename for both daily and monthly naming conventions."""
    fname = os.path.basename(filepath)
    return pd.to_datetime(fname[-10:-4], format='%Y%m')


def get_dataset_paths(t_variable):
    return {'ERA5L': fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\Input_rasters\ERA5L",
    'HAR': fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\Input_rasters\HAR",
    'CHIRTS':fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\Input_rasters\CHIRTS",
    'CHELSA':fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\Input_rasters\CHELSA",
    'IK': fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\Input_rasters\IK",
    'DS': fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\QRF_Products\{t_variable}\Deterministic"
    }

def build_dataset_results(dataset_paths, datasets, t_variable):
    """Build {dataset}_files dict pre-filtered to analysis period."""
    dataset_results = {}
    for dataset in datasets:
        if dataset in ['ERA5L', 'HAR', 'CHIRTS', 'CHELSA']:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{dataset}_{t_variable}_average_*.tif")))
        elif dataset == 'IK':
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{dataset.lower()}_{t_variable}_average_*.tif")))
        else:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{t_variable}_QRFDe_*.tif")))
        filtered = []
        for f in files:
            try:
                d = extract_date(f, dataset)
                if start_date <= d <= end_date:
                    filtered.append(f)
            except Exception as e:
                print(f"  Skipping {os.path.basename(f)}: {e}")
        dataset_results[f'{dataset}_files'] = filtered
        print(f"{dataset}: {len(filtered)} files in period")
    return dataset_results

In [5]:
def _sample_file(file, dataset, stn_points, unique_stn, t_variable):
    """Sample all station points from a single raster file."""
    date_obj = extract_date(file, dataset)
    try:
        with rasterio.open(file) as src:
            if src.crs.to_epsg() != 4326:
                transformer = Transformer.from_crs(
                    "EPSG:4326", src.crs, always_xy=True
                )
                pts = [transformer.transform(x, y) for x, y in stn_points]
            else:
                pts = stn_points

            sampled = list(src.sample(pts))

        return [
            {
                'date':              date_obj,
                'stn_id':            unique_stn.iloc[i]['index_no'],
                f'{dataset}_{t_variable}':   v[0],
            }
            for i, v in enumerate(sampled)
        ]
    except Exception as e:
        print(f"  Error processing {os.path.basename(file)}: {e}")
        return []


def extract_monthly_datasets(dataset_results, stn_points, unique_stn, datasets, t_variable):
    """
    For already-monthly datasets
      Sample each file concurrently and merge at monthly level.
    """
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} monthly files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            for result in executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn, t_variable), files
            ):
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        df_dataset['date'] = pd.to_datetime(df_dataset['date'])
        df_dataset = df_dataset.assign(
            year  = lambda d: d['date'].dt.year,
            month = lambda d: d['date'].dt.month
        ).drop(columns='date')

        df_final = (
            df_dataset if df_final is None
            else pd.merge(df_final, df_dataset,
                          on=['stn_id', 'year', 'month'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")

    return df_final

In [6]:
# Build dataset results - Tmax
tmax_results = build_dataset_results(get_dataset_paths("tmax"), datasets, "tmax")

df_grid_tmax = extract_monthly_datasets(
    tmax_results, stn_points_tmax, unique_stn_tmax, datasets, "tmax" 
)

df_validation_tmax = pd.merge(
    insitu_tmax_monthly_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmax,
    on=['stn_id', 'year', 'month'],
    how='inner'
)
df_validation_tmax.to_csv(os.path.join(output_dir, "Tmax_validation_df.csv"), index = False)

# Build dataset results - Tmin
tmin_results = build_dataset_results(get_dataset_paths("tmin"), datasets, "tmin")

df_grid_tmin = extract_monthly_datasets(
    tmin_results, stn_points_tmin, unique_stn_tmin, datasets, "tmin" 
)

df_validation_tmin = pd.merge(
    insitu_tmin_monthly_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmin,
    on=['stn_id', 'year', 'month'],
    how='inner'
)
df_validation_tmin.to_csv(os.path.join(output_dir, "Tmin_validation_df.csv"), index = False)

ERA5L: 297 files in period
HAR: 297 files in period
CHIRTS: 297 files in period
CHELSA: 297 files in period
IK: 297 files in period
DS: 297 files in period
Processing ERA5L (297 monthly files)...
  → merged shape: (3564, 4)
Processing HAR (297 monthly files)...
  → merged shape: (3564, 5)
Processing CHIRTS (297 monthly files)...
  → merged shape: (3564, 6)
Processing CHELSA (297 monthly files)...
  → merged shape: (3564, 7)
Processing IK (297 monthly files)...
  → merged shape: (3564, 8)
Processing DS (297 monthly files)...
  → merged shape: (3564, 9)
ERA5L: 297 files in period
HAR: 297 files in period
CHIRTS: 297 files in period
CHELSA: 297 files in period
IK: 297 files in period
DS: 297 files in period
Processing ERA5L (297 monthly files)...
  → merged shape: (3564, 4)
Processing HAR (297 monthly files)...
  → merged shape: (3564, 5)
Processing CHIRTS (297 monthly files)...
  → merged shape: (3564, 6)
Processing CHELSA (297 monthly files)...
  → merged shape: (3564, 7)
Processing IK 

In [10]:
def normalize_column(col):
    if col.max() == col.min():
        return pd.Series(0, index=col.index)
    return (col - col.min()) / (col.max() - col.min())
          
def calculate_accuracy(dataframe_temp, t_variable, datasets_list):
    datasets_metrics = []

    for dataset in datasets_list:
        dataset_accuracy = {'dataset': dataset}

        grid_col = f'{dataset}_{t_variable}'
        insitu_col = f'{t_variable}_average'
        
        if grid_col not in dataframe_temp.columns:
            print(f"Skipping {dataset} - column not found.")
            continue

        temp_grid_array = dataframe_temp[grid_col].values
        temp_insitu_array = dataframe_temp[insitu_col].values
    
        valid_indices = np.logical_and.reduce([
            ~np.isnan(temp_grid_array),
            ~np.isnan(temp_insitu_array)
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        if observations == 0:
            continue

        temp_grid_array = temp_grid_array[valid_indices]
        temp_insitu_array = temp_insitu_array[valid_indices]

        dataset_accuracy['correlation'] = he.spearman_r(temp_grid_array, temp_insitu_array)
        dataset_accuracy['nrmse'] = he.nrmse_mean(temp_grid_array, temp_insitu_array)
        dataset_accuracy['rsd'] = (np.std(temp_grid_array)) / (np.std(temp_insitu_array))

        datasets_metrics.append(dataset_accuracy)

    df_accuracy = pd.DataFrame(datasets_metrics)

    df_accuracy['diff_rsd'] = (df_accuracy['rsd'] - 1).abs()
    
    df_accuracy_normalized = df_accuracy.copy()

    metrics = ['correlation', 'nrmse', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    df_accuracy_normalized['cpi'] = (df_accuracy_normalized['correlation'] + 
                                     df_accuracy_normalized['nrmse'] + 
                                     df_accuracy_normalized['diff_rsd'])
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(
        df_accuracy[['dataset','correlation','nrmse', 'rsd', 'observations_n']], 
        on=['dataset'], 
        how='inner'
    )
    return df_accuracy_final


In [11]:
results_tmax = calculate_accuracy(df_validation_tmax, "tmax", datasets)
results_tmax

,dataset,cpi,cpi_rank,correlation,nrmse,rsd,observations_n
0,ERA5L,2.416099,5.0,0.862745,0.203209,1.229267,3458
1,HAR,3.000000,6.0,0.840012,0.204574,1.378845,3458
2,CHIRTS,0.193989,2.0,0.952255,0.065343,0.998373,3458
3,CHELSA,1.057759,4.0,0.946042,0.130620,1.145363,3458
4,IK,0.079554,1.0,0.967513,0.054165,1.031636,3458
5,DS,0.202028,3.0,0.956665,0.061392,1.027612,3458


In [12]:
results_tmin = calculate_accuracy(df_validation_tmin, "tmin", datasets)
results_tmin

,dataset,cpi,cpi_rank,correlation,nrmse,rsd,observations_n
0,ERA5L,2.701421,6.0,0.936895,0.204821,1.140282,3387
1,HAR,2.337908,5.0,0.918596,0.181326,1.088919,3387
2,CHIRTS,0.859807,3.0,0.971783,0.162946,1.038347,3387
3,CHELSA,1.603771,4.0,0.956816,0.151184,1.106173,3387
4,IK,0.261178,2.0,0.978837,0.100599,0.954651,3387
5,DS,0.000000,1.0,0.979885,0.090579,0.972249,3387
